In [1]:
import pandas as pd
import numpy as np
import re
import gensim
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report


C:\Users\RIHAB-PC\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\RIHAB-PC\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\RIHAB-PC\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [13]:
# Load dataset
train_df = pd.read_csv('train (2).csv')  # Adjust the path as needed
test_df = pd.read_csv('test (2).csv')

# Basic text cleaning
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\S+', '', text)  # Remove mentions
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)  # Remove special characters
    return text.lower()

# Clean train and test tweets
train_df['cleaned_text'] = train_df['tweet'].apply(clean_text)
test_df['cleaned_text'] = test_df['tweet'].apply(clean_text)

# Labels (assuming 0 = not offensive, 1 = offensive)
y_train = train_df['class'].values
y_test = test_df['class'].values


In [19]:
# Clean and prepare the tweets
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\S+', '', text)  # Remove mentions
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)  # Remove special characters
    return text.lower()

# Apply cleaning to the 'tweet' column
train_df['cleaned_text'] = train_df['tweet'].apply(clean_text)
test_df['cleaned_text'] = train_df['tweet'].apply(clean_text)

# Labels (0 = non-offensive, 1 = offensive, 2 = hate speech)
y_train = train_df['class'].values


In [21]:
max_words = 10000  # Max number of words to consider
max_len = 100  # Maximum sequence length

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_df['cleaned_text'])

# Convert text to sequences
X = tokenizer.texts_to_sequences(train_df['cleaned_text'])

# Pad sequences to ensure consistent length
X = pad_sequences(X, maxlen=max_len)


In [25]:
# Load Word2Vec embeddings (downloaded file)
word2vec_path = 'GoogleNews-vectors-negative300.bin'  # Adjust the path to Word2Vec model
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

# Create the embedding matrix
embedding_dim = 300
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < max_words:
        try:
            embedding_vector = w2v_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            continue


In [26]:
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False))
model.add(LSTM(128, return_sequences=True))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))  # 3 classes: non-offensive, offensive, hate speech

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


C:\Users\RIHAB-PC\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │       3,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,000,000 (11.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,000,000 (11.44 MB)

In [29]:
history = model.fit(X, y_train, epochs=5, batch_size=64, validation_split=0.2)


Epoch 1/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 66s 225ms/step - accuracy: 0.7928 - loss: 0.5643 - val_accuracy: 0.9007 - val_loss: 0.2930
Epoch 2/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 78s 210ms/step - accuracy: 0.8843 - loss: 0.3237 - val_accuracy: 0.9090 - val_loss: 0.2711
Epoch 3/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 84s 219ms/step - accuracy: 0.8953 - loss: 0.2983 - val_accuracy: 0.9090 - val_loss: 0.2637
Epoch 4/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 56s 226ms/step - accuracy: 0.9051 - loss: 0.2682 - val_accuracy: 0.9095 - val_loss: 0.2670
Epoch 5/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 53s 213ms/step - accuracy: 0.9144 - loss: 0.2385 - val_accuracy: 0.9097 - val_loss: 0.2579


In [57]:
history = model.fit(X, y_train, epochs=2, batch_size=64, validation_split=0.2)


Epoch 1/2
248/248 ━━━━━━━━━━━━━━━━━━━━ 110s 442ms/step - accuracy: 0.9198 - loss: 0.2205 - val_accuracy: 0.9052 - val_loss: 0.2698
Epoch 2/2
248/248 ━━━━━━━━━━━━━━━━━━━━ 111s 447ms/step - accuracy: 0.9289 - loss: 0.1979 - val_accuracy: 0.9060 - val_loss: 0.2759


In [59]:
history = model.fit(X, y_train, epochs=4, batch_size=64, validation_split=0.2)


Epoch 1/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 105s 424ms/step - accuracy: 0.9383 - loss: 0.1615 - val_accuracy: 0.9024 - val_loss: 0.3036
Epoch 2/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 101s 405ms/step - accuracy: 0.9481 - loss: 0.1428 - val_accuracy: 0.9014 - val_loss: 0.3250
Epoch 3/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 141s 401ms/step - accuracy: 0.9588 - loss: 0.1131 - val_accuracy: 0.9009 - val_loss: 0.3381
Epoch 4/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 98s 395ms/step - accuracy: 0.9684 - loss: 0.0926 - val_accuracy: 0.8903 - val_loss: 0.3535


In [61]:
history = model.fit(X, y_train, epochs=4, batch_size=64, validation_split=0.2)


Epoch 1/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 101s 405ms/step - accuracy: 0.9712 - loss: 0.0844 - val_accuracy: 0.8971 - val_loss: 0.3815
Epoch 2/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 142s 404ms/step - accuracy: 0.9792 - loss: 0.0636 - val_accuracy: 0.8863 - val_loss: 0.4098
Epoch 3/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 142s 404ms/step - accuracy: 0.9802 - loss: 0.0571 - val_accuracy: 0.8911 - val_loss: 0.4576
Epoch 4/4
248/248 ━━━━━━━━━━━━━━━━━━━━ 101s 409ms/step - accuracy: 0.9857 - loss: 0.0458 - val_accuracy: 0.8946 - val_loss: 0.4980


In [33]:
# Evaluate on the training data or validation set
train_accuracy = model.evaluate(X, y_train)
print("Training Accuracy: ", train_accuracy)


620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - accuracy: 0.9282 - loss: 0.2011
Training Accuracy:  [0.2154649794101715, 0.9235347509384155]


In [41]:
# Save the model
model.save('pretrained_toxic_tweet_detection_model.keras')


In [37]:
test_df.head()

,count,hate_speech_count,offensive_language_count,neither_count,tweet,cleaned_text
0,3,0,3,0,934 8616\ni got a missed call from yo bitch,rt well how else will white ppl get us to fo...
1,3,0,3,0,RT @KINGTUNCHI_: Fucking with a bad bitch you ...,funny thing isits not just the people doing it...
2,3,0,1,2,RT @eanahS__: @1inkkofrosess lol my credit ain...,rt nigga messed with the wrong bitch 128557...
3,3,0,3,0,RT @Maxin_Betha Wipe the cum out of them faggo...,bitch ass nigggaaa
4,3,1,2,0,Niggas cheat on they bitch and don't expect no...,so that real bitch


In [39]:
train_df.head()

,count,hate_speech_count,offensive_language_count,neither_count,tweet,class,cleaned_text
0,3,2,0,1,RT @FunSizedYogi: @TheBlackVoice well how else...,0,rt well how else will white ppl get us to fo...
1,3,0,0,3,Funny thing is....it's not just the people doi...,2,funny thing isits not just the people doing it...
2,3,0,3,0,"RT @winkSOSA: ""@AintShitSweet__: ""@Rakwon_OGOD...",1,rt nigga messed with the wrong bitch 128557...
3,3,0,3,0,@Jbrendaro30 @ZGabrail @ramsin1995 @GabeEli8 @...,1,bitch ass nigggaaa
4,6,0,6,0,S/o that real bitch,1,so that real bitch


In [47]:
# Load the test dataset
test_data = pd.read_csv('test (2).csv')  # Adjust the path accordingly


In [49]:
# Clean the test data (same function as used for training data)
test_data['cleaned_text'] = test_data['tweet'].apply(clean_text)

# Tokenize the test data
X_test = tokenizer.texts_to_sequences(test_data['cleaned_text'])

# Pad sequences to ensure consistent length (same as training data)
X_test = pad_sequences(X_test, maxlen=max_len)


In [51]:
# Make predictions on the test data
test_predictions = model.predict(X_test)

# Get the predicted class (0 = non-offensive, 1 = offensive, 2 = hate speech)
predicted_classes = np.argmax(test_predictions, axis=1)


155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step


In [55]:
# Prepare the submission file

test_data['id'] = test_data.index

submission = pd.DataFrame({
    'id': test_data['id'],  # Use the 'id' column from the test data
    'sentiment': predicted_classes  # Predicted sentiment class
})

# Save the submission file
submission.to_csv('submission.csv', index=False)
print("done")

done


In [35]:
# Assuming you have test data (X_test) to make predictions on
test_predictions = model.predict(X_test)
submission = pd.DataFrame({'id': test_df['id'], 'sentiment': np.argmax(test_predictions, axis=1)})

# Save the submission file
submission.to_csv('submission.csv', index=False)


NameError: name 'X_test' is not defined

In [ ]:
max_words = 10000  # Max number of words to consider
max_len = 100  # Maximum sequence length

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(data['cleaned_text'])

# Convert text to sequences
X = tokenizer.texts_to_sequences(data['cleaned_text'])

# Pad sequences to ensure consistent length
X = pad_sequences(X, maxlen=max_len)


In [ ]:
max_words = 10000  # Max number of words to consider
max_len = 100  # Maximum sequence length

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_df['cleaned_text'])

# Convert text to sequences
X_train = tokenizer.texts_to_sequences(train_df['cleaned_text'])
X_test = tokenizer.texts_to_sequences(test_df['cleaned_text'])

# Pad sequences to ensure consistent length
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)


In [ ]:
# Load Word2Vec embeddings (downloaded file)
word2vec_path = 'path_to_word2vec.bin'  # Adjust the path to Word2Vec model
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

# Create the embedding matrix
embedding_dim = 300
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < max_words:
        try:
            embedding_vector = w2v_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            continue


In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False))
model.add(LSTM(128, return_sequences=True))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))


In [ ]:
# Predict the test data
y_pred = (model.predict(X_test) > 0.5).astype('int32')

# Print the classification report
print(classification_report(y_test, y_pred))


In [ ]:
model.save('toxic_tweet_detection_model.h5')


In [ ]:
# Prepare submission
test_predictions = model.predict(X_test)
submission = pd.DataFrame({'id': test_df['id'], 'sentiment': (test_predictions > 0.5).astype(int)})

# Save submission file
submission.to_csv('submission.csv', index=False)
